# Food Safety Intelligence — Dataset Overview

**Goal of this notebook:** load and lightly profile the four public datasets that power our Chicago food-safety risk model, so the team can see what we have to work with on day 1.

| # | Dataset | Why it's in the model |
|---|---|---|
| 1 | Chicago **Food Inspections** (`4ijn-s7e5`) | Ground-truth labels (Pass / Fail / violations) — the prediction target lives here |
| 2 | Chicago **311 Service Requests** (`v6vf-nfxy`) | Neighborhood signals: rodents, sanitation, restaurant complaints, illegal dumping |
| 3 | Chicago **Business Licenses — Current Active** (`uupf-x98q`) | Today's universe of licensed establishments (with lat/lon, neighborhood) |
| 4 | Chicago **Business Licenses — Historical** (`vgg9-bn8p`) | License history: when each license started, expired, renewed, status changes |

**How to read this notebook:** every dataset section follows the same shape — *fetch → cache → peek → profile*. We cache each pull to `../data/raw/*.parquet` so re-runs are instant and teammates can work offline.

**Runtime:** ~3–6 minutes the first time (network-bound), seconds on every re-run.

## 0. Setup

Shared imports and the single fetch helper every section uses. All four datasets live on the same Socrata (SODA) API, so one paginating function covers them all.

In [ ]:
from pathlib import Path
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Cache directory — relative to this notebook's location.
DATA_DIR = Path('../data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# All four datasets are served by the City of Chicago Socrata portal.
SODA_BASE = 'https://data.cityofchicago.org/resource'

DATASETS = {
    'inspections':         '4ijn-s7e5',   # Food inspections
    'complaints_311':      'v6vf-nfxy',   # 311 service requests
    'licenses_current':    'uupf-x98q',   # Business licenses — current active
    'licenses_historical': 'vgg9-bn8p',   # Business licenses — historical
}

In [ ]:
def fetch_soda(dataset_id, where=None, select=None, order=None,
               page_size=50_000, max_pages=200, timeout=180,
               max_retries=4, verbose=True):
    """Paginate a SODA endpoint with retry + backoff and return a single DataFrame.

    Resilience matters: a single 120s timeout on page 11 would kill 500k rows
    of already-fetched data. We now:
      - bump per-request timeout to 180s
      - retry transient ReadTimeout / ConnectionError with exponential backoff
      - request gzip so big payloads come back faster

    For datasets where the *offset itself* is the bottleneck (311 past ~500k rows),
    use fetch_soda_keyset below — it pages by a date cursor instead of $offset.
    """
    url = f'{SODA_BASE}/{dataset_id}.json'
    frames = []
    for page in range(max_pages):
        params = {'$limit': page_size, '$offset': page * page_size}
        if where:  params['$where']  = where
        if select: params['$select'] = select
        if order:  params['$order']  = order

        for attempt in range(max_retries):
            try:
                t0 = time.time()
                r = requests.get(url, params=params, timeout=timeout,
                                 headers={'Accept-Encoding': 'gzip'})
                r.raise_for_status()
                elapsed = time.time() - t0
                break
            except (requests.Timeout, requests.ConnectionError) as e:
                if attempt == max_retries - 1:
                    raise
                wait = 5 * (2 ** attempt)  # 5, 10, 20, 40s
                if verbose:
                    print(f'  page {page+1} attempt {attempt+1} failed ({type(e).__name__}); retry in {wait}s')
                time.sleep(wait)

        chunk = pd.DataFrame(r.json())
        if chunk.empty:
            break
        frames.append(chunk)
        if verbose:
            cum = sum(len(f) for f in frames)
            print(f'  page {page+1}: +{len(chunk):,} rows (cum {cum:,}, {elapsed:.1f}s)')
        if len(chunk) < page_size:
            break
        time.sleep(0.2)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _safe_dedupe(df, dedupe_on=None, verbose=True):
    """Dedupe robust to unhashable columns (e.g. SODA's nested 'location' dicts).

    - If `dedupe_on` is given, dedupe on that key column (fast + unambiguous).
    - Else dedupe on the subset of hashable columns (drops dicts/lists from the key).
    """
    if dedupe_on:
        return df.drop_duplicates(subset=[dedupe_on])
    # Sample-check the first 200 rows to find unhashable columns cheaply.
    sample = df.head(200)
    unhashable = [
        c for c in df.columns
        if sample[c].dropna().apply(lambda x: isinstance(x, (dict, list))).any()
    ]
    if unhashable:
        if verbose:
            print(f'  dedupe: ignoring unhashable cols {unhashable}')
        hashable = [c for c in df.columns if c not in unhashable]
        return df.drop_duplicates(subset=hashable)
    return df.drop_duplicates()


def fetch_soda_keyset(dataset_id, cursor_col, cursor_start,
                      where_extra=None, page_size=50_000, max_pages=500,
                      timeout=180, max_retries=4, dedupe_on=None,
                      shard_dir=None, verbose=True):
    """Keyset pagination — pages by `cursor_col >= last_value` instead of $offset.

    Why: $offset is O(offset) on Socrata. Past ~500k matching rows the server
    starts timing out. Keyset pagination keeps every page fast regardless of depth.

    Resilience: if `shard_dir` is given, each page is written to a separate parquet
    shard there *immediately after fetch*. If anything later fails (e.g. dedupe),
    the next call resumes from the highest cursor value already on disk — we don't
    re-fetch 15 minutes of data.

    Dedupe: when many rows share the same cursor value across a page boundary,
    `>=` would re-fetch them, so we always dedupe at the end. `dedupe_on` lets
    the caller name a primary key column (e.g. 'sr_number') for cleaner semantics
    when the table has unhashable nested columns.
    """
    url = f'{SODA_BASE}/{dataset_id}.json'
    cursor = cursor_start
    shards = []

    # Resume from existing shards if any exist.
    if shard_dir is not None:
        shard_dir = Path(shard_dir)
        shard_dir.mkdir(parents=True, exist_ok=True)
        existing = sorted(shard_dir.glob('page_*.parquet'))
        if existing:
            shards = existing
            last = pd.read_parquet(existing[-1])
            cursor = str(last[cursor_col].max())
            if verbose:
                cum = sum(len(pd.read_parquet(s)) for s in shards)
                print(f'  resuming from {len(shards)} shard(s): {cum:,} rows, cursor → {cursor[:19]}')

    start_page = len(shards)
    for page in range(start_page, max_pages):
        where_parts = [f"{cursor_col} >= '{cursor}'"]
        if where_extra:
            where_parts.append(f'({where_extra})')
        params = {
            '$where': ' AND '.join(where_parts),
            '$order': f'{cursor_col} ASC',
            '$limit': page_size,
        }
        for attempt in range(max_retries):
            try:
                t0 = time.time()
                r = requests.get(url, params=params, timeout=timeout,
                                 headers={'Accept-Encoding': 'gzip'})
                r.raise_for_status()
                elapsed = time.time() - t0
                break
            except (requests.Timeout, requests.ConnectionError) as e:
                if attempt == max_retries - 1:
                    raise
                wait = 5 * (2 ** attempt)
                if verbose:
                    print(f'  page {page+1} attempt {attempt+1} failed ({type(e).__name__}); retry in {wait}s')
                time.sleep(wait)

        chunk = pd.DataFrame(r.json())
        if chunk.empty:
            break

        # Persist this page to disk immediately so a downstream crash doesn't lose it.
        if shard_dir is not None:
            shard_path = shard_dir / f'page_{page:04d}.parquet'
            chunk.astype({c: 'string' for c in chunk.select_dtypes('object').columns}) \
                 .to_parquet(shard_path, index=False)
            shards.append(shard_path)

        if verbose:
            cum_pages = page + 1
            print(f'  page {cum_pages}: +{len(chunk):,} rows (cursor={str(cursor)[:10]}, {elapsed:.1f}s)')
        if len(chunk) < page_size:
            break
        cursor = chunk[cursor_col].iloc[-1]
        time.sleep(0.2)

    if shards:
        out = pd.concat([pd.read_parquet(s) for s in shards], ignore_index=True)
    else:
        return pd.DataFrame()

    before = len(out)
    out = _safe_dedupe(out, dedupe_on=dedupe_on, verbose=verbose)
    if verbose and len(out) != before:
        print(f'  after dedupe: {len(out):,} rows (removed {before - len(out):,})')

    # On clean success, clean up the shard dir.
    if shard_dir is not None:
        for s in shards:
            s.unlink(missing_ok=True)
        try:
            shard_dir.rmdir()
        except OSError:
            pass  # not empty for some reason — leave it

    return out


def load_or_fetch(name, fetch_fn):
    """Load a cached parquet if present, otherwise run fetch_fn() and cache it.

    Teammates: delete the parquet under data/raw/ to force a refresh.
    """
    cache = DATA_DIR / f'{name}.parquet'
    if cache.exists():
        print(f'✓ loading {name} from cache: {cache}')
        return pd.read_parquet(cache)
    print(f'↻ fetching {name} from API...')
    df = fetch_fn()
    df = df.astype({c: 'string' for c in df.select_dtypes('object').columns})
    df.to_parquet(cache, index=False)
    print(f'  saved → {cache}  ({len(df):,} rows)')
    return df


def peek(df, name):
    """Standard one-page profile we print for every dataset."""
    print(f'=== {name} ===')
    print(f'shape:   {df.shape[0]:,} rows × {df.shape[1]} cols')
    print(f'memory:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    print(f'columns: {list(df.columns)}')
    print('\n— missing values (top 10) —')
    miss = df.isna().mean().sort_values(ascending=False).head(10)
    print((miss * 100).round(2).astype(str) + '%')
    print('\n— head(3) —')
    return df.head(3)

## 1. Food Inspections — full history (2010 → present)

Source: https://data.cityofchicago.org/Health-Human-Services/Food-Inspections/4ijn-s7e5

This is the **labeled outcomes** table — every row is one inspection visit. Key columns:
- `inspection_id`, `license_` — joins to business licenses
- `inspection_date`, `inspection_type` (Canvass / Complaint / License / Re-Inspection)
- `results` — Pass / Pass w/ Conditions / Fail / Out of Business / No Entry — **the prediction target**
- `violations` — `|`-separated text, each chunk prefixed with a numbered code (1–29 are priority / 'critical' violations)
- `risk`, `facility_type`, `latitude`, `longitude`

In [ ]:
def _fetch_inspections():
    return fetch_soda(
        DATASETS['inspections'],
        order='inspection_date ASC, inspection_id ASC',
    )

inspections = load_or_fetch('inspections', _fetch_inspections)

# Drop Socrata-internal computed_region columns + nested location.
inspections = inspections.loc[:, ~inspections.columns.str.startswith(':@computed_region')]
if 'location' in inspections.columns:
    inspections = inspections.drop(columns=['location'])

# Coerce types — everything comes off SODA as string.
inspections['inspection_date'] = pd.to_datetime(inspections['inspection_date'])
inspections['latitude']  = pd.to_numeric(inspections['latitude'],  errors='coerce')
inspections['longitude'] = pd.to_numeric(inspections['longitude'], errors='coerce')

peek(inspections, 'Food Inspections')

In [ ]:
print('Date range:', inspections['inspection_date'].min().date(),
      '→', inspections['inspection_date'].max().date())
print(f'\nUnique licenses (facilities): {inspections["license_"].nunique():,}')
print(f'Inspections per year (last 5):')
print(inspections.assign(year=inspections['inspection_date'].dt.year)
      .groupby('year').size().tail(5).to_string())

print('\n--- results distribution ---')
print(inspections['results'].value_counts(dropna=False).to_string())

In [ ]:
# Quick visual — inspections per month over the full history.
ax = (inspections.assign(m=inspections['inspection_date'].dt.to_period('M'))
      .groupby('m').size().plot(title='Food inspections per month (2010 → present)'))
ax.set_xlabel(''); ax.set_ylabel('inspections')
plt.tight_layout(); plt.show()

### Inspection outcome mix and modeling relevance

The inspection table is our label source, so before joining any external data we need to understand how many rows represent true food-safety outcomes versus administrative/non-inspection outcomes. This also helps explain why later notebooks drop or mask rows like `No Entry`, `Out of Business`, and `Not Ready` before training.

In [ ]:
# Outcome mix: which inspection results are modelable food-safety outcomes?
MODELABLE_RESULTS = {'Pass', 'Pass w/ Conditions', 'Fail'}

result_summary = (
    inspections['results']
    .fillna('Missing')
    .value_counts()
    .rename_axis('results')
    .reset_index(name='n')
)

result_summary['share'] = result_summary['n'] / result_summary['n'].sum()
result_summary['is_modelable'] = result_summary['results'].isin(MODELABLE_RESULTS)

result_summary

In [ ]:
# Yearly inspection volume and outcome rates among modelable inspections.
MODELABLE_RESULTS = {'Pass', 'Pass w/ Conditions', 'Fail'}

inspection_year_summary = (
    inspections
    .assign(
        year=inspections['inspection_date'].dt.year,
        is_fail=inspections['results'].eq('Fail'),
        is_conditional=inspections['results'].eq('Pass w/ Conditions'),
        is_modelable=inspections['results'].isin(MODELABLE_RESULTS),
    )
    .groupby('year')
    .agg(
        n_inspections=('inspection_id', 'size'),
        n_modelable=('is_modelable', 'sum'),
        modelable_share=('is_modelable', 'mean'),
        unique_licenses=('license_', 'nunique'),
    )
)

modelable_by_year = (
    inspections[inspections['results'].isin(MODELABLE_RESULTS)]
    .assign(
        year=lambda d: d['inspection_date'].dt.year,
        is_fail=lambda d: d['results'].eq('Fail'),
        is_conditional=lambda d: d['results'].eq('Pass w/ Conditions'),
    )
    .groupby('year')
    .agg(
        fail_rate=('is_fail', 'mean'),
        conditional_pass_rate=('is_conditional', 'mean'),
    )
)

inspection_year_summary = inspection_year_summary.join(modelable_by_year)

inspection_year_summary.tail(10).round(3)

In [ ]:
# Visual: outcome rate over time.
ax = inspection_year_summary[['fail_rate', 'conditional_pass_rate']].plot(
    marker='o',
    title='Inspection outcome rates by year'
)
ax.set_xlabel('')
ax.set_ylabel('Share of inspections')
plt.tight_layout()
plt.show()

**Inspection EDA takeaways**

- The inspection data is usable as a label source, but not every `results` value is a trainable food-safety outcome.
- `Fail` and `Pass w/ Conditions` rates over time are useful drift checks before modeling.
- The model should avoid treating administrative outcomes such as `No Entry` or `Out of Business` as clean negatives.

## 2. 311 Service Requests — food-safety-relevant complaints

Source: https://data.cityofchicago.org/Service-Requests/311-Service-Requests/v6vf-nfxy

The full 311 table has ~14M rows across 100+ request types — almost all of which are irrelevant to food safety (potholes, traffic signals, etc.). We pull only the types that plausibly correlate with food-safety risk, since 2019 (matching our modeling window).

Two tiers of relevance:
- **Direct food-safety**: `Restaurant Complaint`, `Pushcart Food Vendor Complaint`
- **Ambient sanitation signal** (rodents, garbage, dumping): hyperlocal proxies for unsanitary conditions near food establishments

In [ ]:
# SR types we care about. Discovered by querying $select=sr_type,count(*) on the API.
RELEVANT_SR_TYPES = [
    'Restaurant Complaint',
    'Pushcart Food Vendor Complaint',
    'Rodent Baiting/Rat Complaint',
    'Sanitation Code Violation',
    'Garbage Cart Maintenance',
    'Missed Garbage Pick-Up Complaint',
    'Fly Dumping Complaint',
    'Dead Animal Pick-Up Request',
]

def _fetch_311():
    # 311 since 2019 across these 8 types is ~1.1M rows. Two things we need:
    # 1) Keyset pagination on created_date (offset-pagination times out past ~500k).
    # 2) Dedupe on sr_number (each SR has a unique number) — the table contains a
    #    nested 'location' dict column that breaks the default drop_duplicates().
    # 3) shard_dir → each page is written to disk as it arrives, so if anything
    #    later fails we resume from where we left off instead of re-pulling 15 min.
    quoted = ', '.join(f"'{t}'" for t in RELEVANT_SR_TYPES)
    return fetch_soda_keyset(
        DATASETS['complaints_311'],
        cursor_col='created_date',
        cursor_start='2019-01-01T00:00:00',
        where_extra=f'sr_type IN ({quoted})',
        dedupe_on='sr_number',
        shard_dir=DATA_DIR / '_partial_complaints_311',
    )

complaints = load_or_fetch('complaints_311', _fetch_311)
complaints['created_date'] = pd.to_datetime(complaints['created_date'])
if 'closed_date' in complaints.columns:
    complaints['closed_date'] = pd.to_datetime(complaints['closed_date'], errors='coerce')
if 'latitude' in complaints.columns:
    complaints['latitude']  = pd.to_numeric(complaints['latitude'],  errors='coerce')
    complaints['longitude'] = pd.to_numeric(complaints['longitude'], errors='coerce')

peek(complaints, '311 Complaints (food-safety-relevant)')

### 311 API note

The 311 pull is the slowest fetch in this notebook because the filtered table is still large. The keyset pagination helper writes temporary shards during download, so interrupted pulls can be recovered locally, but the committed notebook keeps the full API fetch as the source of truth.

In [ ]:
print('--- sr_type counts ---')
print(complaints['sr_type'].value_counts().to_string())

print('\n--- complaints per year ---')
by_year = complaints.assign(yr=complaints['created_date'].dt.year).groupby('yr').size()
print(by_year.to_string())

# Stacked monthly trend by sr_type — does any category spike around weather / COVID?
monthly = (complaints.assign(m=complaints['created_date'].dt.to_period('M').dt.to_timestamp())
           .groupby(['m', 'sr_type']).size().unstack(fill_value=0))
ax = monthly.plot.area(figsize=(12, 5), title='311 food-safety-relevant complaints / month',
                       alpha=0.85, linewidth=0)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=8)
ax.set_xlabel(''); plt.tight_layout(); plt.show()

### 311 complaint signal quality

311 complaints are not outcomes. They are contextual signals that may capture nearby sanitation pressure, reporting intensity, and direct food-related complaint activity. Before using them as spatial features, we need to check whether the request types and coordinates are complete enough to support distance-based joins.

In [ ]:
# Share of complaints by request type.
complaint_type_summary = (
    complaints['sr_type']
    .value_counts()
    .rename_axis('sr_type')
    .reset_index(name='n')
)

complaint_type_summary['share'] = (
    complaint_type_summary['n'] / complaint_type_summary['n'].sum()
)

complaint_type_summary

In [ ]:
# Coordinate completeness matters for spatial joins to restaurants.
coord_cols = [c for c in ['latitude', 'longitude'] if c in complaints.columns]

if coord_cols:
    complaint_coord_quality = (
        complaints
        .assign(has_coordinates=complaints[coord_cols].notna().all(axis=1))
        .groupby('sr_type')
        .agg(
            n=('sr_type', 'size'),
            coordinate_complete_share=('has_coordinates', 'mean'),
        )
        .sort_values('n', ascending=False)
    )
    display(complaint_coord_quality.round(3))
else:
    print('No latitude/longitude columns found in complaints data.')

In [ ]:
# Direct food complaint vs ambient sanitation proxy.
DIRECT_FOOD_SR_TYPES = {
    'Restaurant Complaint',
    'Pushcart Food Vendor Complaint',
}

complaints_signal = complaints.assign(
    signal_group=complaints['sr_type'].apply(
        lambda x: 'Direct food complaint'
        if x in DIRECT_FOOD_SR_TYPES
        else 'Ambient sanitation proxy'
    )
)

signal_summary = (
    complaints_signal
    .groupby('signal_group')
    .agg(
        n=('sr_type', 'size'),
        first_date=('created_date', 'min'),
        last_date=('created_date', 'max'),
    )
)

signal_summary['share'] = signal_summary['n'] / signal_summary['n'].sum()
signal_summary

**311 EDA takeaways**

- Direct restaurant/pushcart complaints are closest to food-safety risk.
- Rodent, sanitation, garbage, dumping, and dead animal requests are broader neighborhood sanitation proxies.
- Coordinate completeness is important because downstream features will count complaints near a restaurant within a time window.
- 311 volume can reflect resident reporting behavior, so these features should be treated as contextual risk signals rather than ground truth.

## 3. Business Licenses — Current Active

Source: https://data.cityofchicago.org/Community-Economic-Development/Business-Licenses-Current-Active/uupf-x98q

The snapshot of currently-active licenses. We want this for two reasons:
1. **Coordinates and community area** for every active establishment — fills the geographic gaps in the inspections table.
2. **Operator-level entity resolution** via `legal_name` / `account_number` — lets us link one owner to multiple licenses (a powerful feature we discussed).

In [ ]:
def _fetch_licenses_current():
    # Small enough (~80k) to load all and filter in pandas, keeps the loader simple.
    return fetch_soda(
        DATASETS['licenses_current'],
        order='license_number ASC',
    )

licenses_current = load_or_fetch('licenses_current', _fetch_licenses_current)
for col in ['license_start_date', 'expiration_date', 'date_issued']:
    if col in licenses_current.columns:
        licenses_current[col] = pd.to_datetime(licenses_current[col], errors='coerce')
for col in ['latitude', 'longitude']:
    if col in licenses_current.columns:
        licenses_current[col] = pd.to_numeric(licenses_current[col], errors='coerce')

peek(licenses_current, 'Licenses (current active)')

In [ ]:
# Which license_descriptions are food-related? These are the rows we'll actually use
# downstream — restaurants, food prep, taverns, food sellers.
FOOD_LICENSE_KEYWORDS = ['food', 'restaurant', 'tavern', 'liquor', 'kitchen']

desc = licenses_current['license_description'].fillna('').str.lower()
food_mask = desc.str.contains('|'.join(FOOD_LICENSE_KEYWORDS), regex=True, na=False)

food_licenses = licenses_current[food_mask]
print(f'Total active licenses:        {len(licenses_current):,}')
print(f'Food-related active licenses: {len(food_licenses):,} ({food_mask.mean():.1%})')

print('\n--- top food license types ---')
print(food_licenses['license_description'].value_counts().head(15).to_string())

### Current license coverage and active restaurant universe

The current license table helps identify the active food-establishment universe and provides address/location metadata. This is useful for the app view because users will search for currently operating restaurants, but it should not be the only source for historical modeling because older inspected restaurants may no longer appear in the current active snapshot.

In [ ]:
# Current active food-license geography/metadata completeness.
current_license_quality_cols = [
    c for c in ['latitude', 'longitude', 'license_start_date', 'expiration_date', 'legal_name', 'account_number']
    if c in food_licenses.columns
]

current_license_quality = (
    food_licenses[current_license_quality_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename('missing_pct')
    .to_frame()
)

current_license_quality

In [ ]:
# Which active food license types dominate?
top_current_food_license_types = (
    food_licenses['license_description']
    .value_counts()
    .head(12)
)

ax = top_current_food_license_types.sort_values().plot.barh(
    title='Top current active food-related license types'
)
ax.set_xlabel('Active licenses')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Potential operator-level feature: one legal/account entity may own multiple licenses.
if 'account_number' in food_licenses.columns:
    operator_license_counts = (
        food_licenses
        .dropna(subset=['account_number'])
        .groupby('account_number')
        .agg(
            n_active_food_licenses=('license_number', 'nunique'),
            legal_name=('legal_name', 'first') if 'legal_name' in food_licenses.columns else ('account_number', 'first'),
        )
        .sort_values('n_active_food_licenses', ascending=False)
    )

    operator_license_counts.head(10)
else:
    print('account_number not available in current licenses.')

**Current license EDA takeaways**

- Current licenses are useful for active restaurant metadata and app-facing lookup.
- Coordinate and license-date completeness determine how much enrichment we can use downstream.
- `account_number` / `legal_name` may support operator-level features, such as whether one owner has multiple active food licenses.

## 4. Business Licenses — Historical

Source: https://data.cityofchicago.org/Community-Economic-Development/Business-licenses/vgg9-bn8p

The full history of every license issuance, renewal, and status change since 2002. It's ~2M rows — far too big for an "all data" pull — so we filter server-side to food-related licenses only, which is what we'll actually use.

Key fields:
- `license_number` — joins to `license_` in food inspections (the bridge between the two datasets)
- `application_type` (ISSUE / RENEW / C_LOC / C_CAPA) — tells us new vs renewed
- `license_status` (AAI = Active / AAC = Cancelled / REV = Revoked / etc.)
- `license_start_date`, `expiration_date` — for computing **license age** (a Tier-1 feature)

In [ ]:
def _fetch_licenses_historical():
    # Filter server-side to food/restaurant/tavern/liquor licenses to keep this tractable.
    # upper() makes it case-insensitive; like-pattern matching is supported by SoQL.
    where = (
        "(upper(license_description) like '%FOOD%'"
        " OR upper(license_description) like '%RESTAURANT%'"
        " OR upper(license_description) like '%TAVERN%'"
        " OR upper(license_description) like '%LIQUOR%'"
        " OR upper(license_description) like '%KITCHEN%')"
    )
    return fetch_soda(
        DATASETS['licenses_historical'],
        where=where,
        order='date_issued ASC, license_number ASC',
    )

licenses_hist = load_or_fetch('licenses_historical', _fetch_licenses_historical)
for col in ['license_start_date', 'expiration_date', 'date_issued', 'payment_date']:
    if col in licenses_hist.columns:
        licenses_hist[col] = pd.to_datetime(licenses_hist[col], errors='coerce')

peek(licenses_hist, 'Licenses (historical, food-related)')

In [ ]:
print('--- license_description (top 15) ---')
print(licenses_hist['license_description'].value_counts().head(15).to_string())

print('\n--- application_type ---')
print(licenses_hist['application_type'].value_counts().to_string())

print('\n--- license_status (top 10) ---')
print(licenses_hist['license_status'].value_counts().head(10).to_string())

# New food licenses issued per year — proxy for food-establishment turnover
issued = (licenses_hist[licenses_hist['application_type'] == 'ISSUE']
          .assign(yr=licenses_hist['date_issued'].dt.year)
          .groupby('yr').size())
ax = issued.tail(15).plot.bar(title='New food-related licenses ISSUED per year')
ax.set_xlabel(''); ax.set_ylabel('count'); plt.tight_layout(); plt.show()

### Historical license features

The historical license table turns a point-in-time inspection into richer business context. The most important feature candidates are license age, renewal history, and status history. These are especially useful because they are available before a future inspection outcome and can be computed without leaking the label.

In [ ]:
# Historical license depth: how many history rows exist per license?
license_history_depth = (
    licenses_hist
    .groupby('license_number')
    .size()
    .describe()
)

license_history_depth

In [ ]:
licenses_hist_sorted = licenses_hist.sort_values(['license_number', 'date_issued'])

license_age_base = (
    licenses_hist_sorted
    .groupby('license_number')
    .agg(
        first_license_start=('license_start_date', 'min'),
        first_issued=('date_issued', 'min'),
        last_expiration=('expiration_date', 'max'),
        n_license_history_rows=('license_number', 'size'),
        n_application_types=('application_type', 'nunique'),
        latest_status=('license_status', 'last'),
    )
)

license_age_base.head()

In [ ]:
# Distribution of historical depth per license.
ax = (
    license_age_base['n_license_history_rows']
    .clip(upper=20)
    .plot.hist(
        bins=20,
        title='Historical license rows per license, clipped at 20'
    )
)
ax.set_xlabel('History rows per license')
plt.tight_layout()
plt.show()

**Historical license EDA takeaways**

- Historical licenses can support leak-free business-history features such as license age and renewal count.
- Most licenses should have only a small number of history rows, while multi-row licenses capture renewals, changes, or status updates.
- These features are safer than inspection-result-derived features because they are business metadata rather than future outcomes.

## 5. How the four datasets connect

These are the joins we'll rely on in the feature pipeline. Each has a known sharp edge — calling them out now saves a debugging week later.

| From | → | To | Key | Gotcha |
|---|---|---|---|---|
| Food Inspections | → | Business Licenses (hist) | `license_` ↔ `license_number` | `license_ = '0'` is a placeholder used for unlicensed events — drop before joining |
| Food Inspections | → | Business Licenses (current) | `license_` ↔ `license_number` | Many old inspections are at facilities whose license has since expired — they won't be in *current* |
| Food Inspections | ↔ | 311 Complaints | `latitude`/`longitude` spatial join | 311 lat/lon is geocoded from the address and can be coarse (block-centroid) — use a 100–300m buffer, not a literal point match |
| Business Licenses (current) | ↔ | Business Licenses (hist) | `license_number` | Current is a snapshot of *active* rows; historical has every issuance/renewal — join hist's earliest `license_start_date` for **license age**, which is a strong feature |

The next two cells sanity-check the most important join (inspections ↔ historical licenses) so we know what % of inspections we can actually enrich with license metadata.

In [ ]:
# How much overlap exists between inspections.license_ and licenses_hist.license_number?
insp_lic = (inspections.loc[~inspections['license_'].isin(['', '0']), 'license_']
            .dropna().astype(str).unique())
hist_lic = licenses_hist['license_number'].dropna().astype(str).unique()

insp_set = set(insp_lic)
hist_set = set(hist_lic)
shared = insp_set & hist_set

print(f'Unique licenses in inspections (food):       {len(insp_set):,}')
print(f'Unique licenses in historical (food filter): {len(hist_set):,}')
print(f'Overlap (inspections joinable to history):   {len(shared):,}')
print(f'Coverage = {len(shared)/len(insp_set):.1%} of inspection-side licenses match')

In [ ]:
# Tiny worked example: take 5 sample inspections, join to historical licenses, show
# the kinds of enrichments we'd get (license_description, dates, status).
sample = inspections[inspections['license_'].isin(list(shared)[:5])][
    ['inspection_id', 'license_', 'dba_name', 'inspection_date', 'results']
].drop_duplicates('license_').head(5)

enrichment = (licenses_hist[licenses_hist['license_number'].isin(sample['license_'])]
              .groupby('license_number')
              .agg(first_issued=('date_issued', 'min'),
                   last_renewal=('date_issued', 'max'),
                   n_history_rows=('license_number', 'size'),
                   license_description=('license_description', 'first'))
              .reset_index().rename(columns={'license_number': 'license_'}))

sample.merge(enrichment, on='license_', how='left')

### Join coverage: license-level and inspection-row-level

The unique-license overlap tells us how many distinct facilities can be enriched. The row-level overlap tells us how much of the actual inspection training table can receive historical license features.

In [ ]:
# Row-level joinability from inspections to historical licenses.
inspection_valid_license = inspections[
    ~inspections['license_'].isin(['', '0'])
].copy()

inspection_valid_license['joins_to_license_history'] = (
    inspection_valid_license['license_'].astype(str).isin(hist_set)
)

row_join_summary = pd.Series({
    'inspection_rows_total': len(inspections),
    'inspection_rows_with_valid_license': len(inspection_valid_license),
    'inspection_rows_joinable_to_history': inspection_valid_license['joins_to_license_history'].sum(),
    'valid_license_row_join_rate': inspection_valid_license['joins_to_license_history'].mean(),
})

row_join_summary.to_frame('value')

In [ ]:
# Join coverage over time.
join_by_year = (
    inspection_valid_license
    .assign(year=inspection_valid_license['inspection_date'].dt.year)
    .groupby('year')
    .agg(
        n_inspection_rows=('inspection_id', 'size'),
        join_rate=('joins_to_license_history', 'mean'),
    )
)

ax = join_by_year['join_rate'].plot(
    marker='o',
    title='Inspection rows joinable to historical licenses by year'
)
ax.set_xlabel('')
ax.set_ylabel('Join rate')
plt.tight_layout()
plt.show()

**Join coverage takeaway**

If row-level join coverage is high, license-history features can be used broadly in the modeling table. If coverage drops in older years, that supports using 2019+ as the training window and treating older records mostly as burn-in/history.

## EDA summary for handoff

This notebook validates that the four source datasets are usable for the MVP modeling pipeline:

1. **Food inspections** provide the outcome source, but only some `results` values are modelable food-safety outcomes.
2. **311 complaints** provide contextual neighborhood and direct complaint signals, but should be treated as noisy proxy features rather than labels.
3. **Current licenses** help define active food establishments and support app-facing lookup.
4. **Historical licenses** support leak-free business-history features such as license age, renewal count, and status history.
5. The key join path is `inspections.license_` → `licenses_hist.license_number`; row-level coverage determines how broadly license features can be used.

## 6. What's next

With the four datasets loaded and cached, the next notebook (`02_feature_engineering.ipynb`) will build the modeling table:

1. **Inspection-level base table** — one row per inspection, target = `Fail` or `has_priority_violation`.
2. **Facility history features** (leak-free): prior fail count, prior critical-violation count, days since last inspection, rolling violation rate.
3. **License features**: license age (from historical), license type, currently-active flag.
4. **Neighborhood 311 features**: counts of rodent/sanitation/restaurant complaints within 300m and 90 days of each inspection.
5. **Geographic context**: community area, ZIP fail-rate (leak-free), spatial-neighbor fail rate.

Re-running this notebook is cheap — every dataset is cached under `data/raw/`. Delete the parquet to force a refresh.